# Effect of training size
**Report section:** Training data size (10 pts)

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = True  # True = 2,000-row debug run, False = full data (use for the report)

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import SEED, set_seed, RESULTS_DIR
from src.data import get_split, load_test, load_unlabeled, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, predict_proba1, plot_roc, savefig,
                          log_result, save_predictions)
set_seed(SEED)

## Data (already cleaned)
`src/data.py` fixes encoding, applies NFC, and drops empty/duplicate rows (there were none).
Case, punctuation and typos are kept. The split is the same for everyone.
**Baseline to beat:** val AUROC 0.956, CV 0.954.

In [ ]:
train_df, val_df = get_split(quick=QUICK)   # 8,000 / 2,000 rows, 50% AI each
test_df = load_test()                        # 3,000 rows (never dropped)
unl_df = load_unlabeled()                    # 3,000 rows, no labels
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)

## Experiments to run
1. **Learning curves** for the best 2–3 pipelines at fractions
   `[0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]` × 3 seeds. Subsample `train_df` with
   `train_test_split(..., stratify=...)` and always score on `val_df`.
2. **Plot** mean ± std val AUROC vs. number of examples (log x-axis).
3. **Discuss:** does the curve flatten, and which model gains most from more data?
4. Bonus: compare against adding pseudo-labelled data (see `notebooks/allen/unlabeled.ipynb`).

**Logging:** `log_result("<experiment>", config, metrics)` writes to `results/`, and
`savefig(fig, "<name>")` writes to `figures/`. Save a final model's predictions with
`save_predictions("<name>", val_df, val_probs, test_df, test_probs)`.

In [ ]:
# your experiments here